In [1]:
# Spark SQL Step-by-Step Explanation

# ------------------------------------------------------------
# 1. Import SparkSession
# ------------------------------------------------------------
from pyspark.sql import SparkSession

In [2]:
# ------------------------------------------------------------
# 2. Create Spark Session
# ------------------------------------------------------------
spark = (
    SparkSession
    .builder
    .appName("Spark SQL")
    .master("local[*]")
    .enableHiveSupport()
    .config("spark.sql.warehouse.dir", "/data/output/spark-warehouse")
    .getOrCreate()
)


In [28]:
# ------------------------------------------------------------
# 3. Define Employee Schema
# ------------------------------------------------------------
_schema = """
first_name string,
last_name string,
job_title string,
dob string,
email string,
phone string,
salary double,
department_id int
"""

In [30]:
# ------------------------------------------------------------
# 4. Read Employee CSV File
# ------------------------------------------------------------
emp = (
    spark.read
    .format("csv")
    .schema(_schema)
    .option("header", True)
    .load("/content/large_employees_50000.csv")
)

# Explanation:
# .format("csv") -> Reads CSV file.
# .schema(_schema) -> Applies predefined schema.
# .option("header", True) -> Uses first row as column names.
# .load(...) -> Loads CSV into DataFrame.

In [32]:
# ------------------------------------------------------------
# 5. Define Department Schema
# ------------------------------------------------------------
_dept_schema = """
department_id int,
department_name string,
description string,
city string,
state string,
country string
"""

In [34]:
# ------------------------------------------------------------
# 6. Read Department CSV File
# ------------------------------------------------------------
dept = (
    spark.read
    .format("csv")
    .schema(_dept_schema)
    .option("header", True)
    .load("/content/large_departments_10000.csv")
)


In [35]:
# ------------------------------------------------------------
# 7. Check Catalog Implementation
# ------------------------------------------------------------
spark.conf.get("spark.sql.catalogImplementation")

# Checks whether Spark uses:
# - in-memory catalog
# OR
# - Hive catalog


'hive'

In [36]:
# ------------------------------------------------------------
# 8. Show Available Databases
# ------------------------------------------------------------
db = spark.sql("show databases")
db.show()

# Displays all available databases.


+---------+
|namespace|
+---------+
|  default|
+---------+



In [37]:

# ------------------------------------------------------------
# 9. Show Tables in Default Database
# ------------------------------------------------------------
spark.sql("show tables in default").show()

# Displays tables available in default database.

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|         |department_filter...|       true|
|         |           dept_view|       true|
|         |            emp_view|       true|
+---------+--------------------+-----------+



In [38]:

# ------------------------------------------------------------
# 10. Create Temporary Views
# ------------------------------------------------------------
emp.createOrReplaceTempView("emp_view")
dept.createOrReplaceTempView("dept_view")

# Temporary views allow SQL queries on DataFrames.


In [57]:
# ------------------------------------------------------------
# 11. Filter Employee Data Using SQL
# ------------------------------------------------------------
emp_filtered = spark.sql("""
    SELECT *
    FROM emp_view
    WHERE department_id = 1
""")

# Filters employees belonging to department_id = 1.

In [58]:

# ------------------------------------------------------------
# 12. Show Filtered Data
# ------------------------------------------------------------
emp_filtered.show()

# Displays filtered records.


+----------+---------+---------+---+-----+-----+------+-------------+
|first_name|last_name|job_title|dob|email|phone|salary|department_id|
+----------+---------+---------+---+-----+-----+------+-------------+
+----------+---------+---------+---+-----+-----+------+-------------+



In [59]:
# ------------------------------------------------------------
# 13. Extract Year from DOB
# ------------------------------------------------------------
emp_temp = spark.sql("""
    SELECT
        e.*,
        date_format(dob, 'yyyy') AS dob_year
    FROM emp_view e
""")

# date_format() extracts year from DOB column.
# Creates a new column called dob_year.

In [60]:

# ------------------------------------------------------------
# 14. Create Another Temporary View
# ------------------------------------------------------------
emp_temp.createOrReplaceTempView("emp_temp_view")

# Registers transformed DataFrame as temp view.


In [61]:

# ------------------------------------------------------------
# 15. Display Updated Data
# ------------------------------------------------------------
spark.sql("SELECT * FROM emp_temp_view").show()

# Shows employee data with dob_year column.

+----------+-----------+---------+----------+----------+-----+--------+-------------+--------+
|first_name|  last_name|job_title|       dob|     email|phone|  salary|department_id|dob_year|
+----------+-----------+---------+----------+----------+-----+--------+-------------+--------+
|    100000|Christopher|Wilkerson|2004-05-01|2022-05-30| 3167| 62385.0|         NULL|    2004|
|    100001|   Jennifer|    Moore|1992-11-24|2025-12-06| 9271| 87074.0|         NULL|    1992|
|    100002|     Julian|    Jones|1992-09-23|2019-10-25| 9654|138568.0|         NULL|    1992|
|    100003|     Thomas|  Roberts|1964-08-26|2018-04-28| 5948| 71807.0|         NULL|    1964|
|    100004|     Robert|    Noble|1987-02-08|2011-05-02| 5094|108778.0|         NULL|    1987|
|    100005|    Brandon|    Myers|1988-07-01|2016-08-25| 4215| 75505.0|         NULL|    1988|
|    100006|  Alexandra|  Hancock|1987-10-05|2025-01-20| 4865|147092.0|         NULL|    1987|
|    100007|      Diana| Thompson|1981-01-03|2014-

In [62]:
# ------------------------------------------------------------
# 16. Join Employee and Department Data
# ------------------------------------------------------------
emp_final = spark.sql("""
    SELECT /*+ BROADCAST(d) */
        e.*,
        d.department_name
    FROM emp_view e
    LEFT OUTER JOIN dept_view d
    ON e.department_id = d.department_id
""")

# LEFT OUTER JOIN keeps all employee records.
# BROADCAST hint improves join performance
# by sending smaller department table to all nodes.

In [63]:


# ------------------------------------------------------------
# 17. Display Final Joined Data
# ------------------------------------------------------------
emp_final.show()

# Shows final joined result.

+----------+-----------+---------+----------+----------+-----+--------+-------------+---------------+
|first_name|  last_name|job_title|       dob|     email|phone|  salary|department_id|department_name|
+----------+-----------+---------+----------+----------+-----+--------+-------------+---------------+
|    100000|Christopher|Wilkerson|2004-05-01|2022-05-30| 3167| 62385.0|         NULL|           NULL|
|    100001|   Jennifer|    Moore|1992-11-24|2025-12-06| 9271| 87074.0|         NULL|           NULL|
|    100002|     Julian|    Jones|1992-09-23|2019-10-25| 9654|138568.0|         NULL|           NULL|
|    100003|     Thomas|  Roberts|1964-08-26|2018-04-28| 5948| 71807.0|         NULL|           NULL|
|    100004|     Robert|    Noble|1987-02-08|2011-05-02| 5094|108778.0|         NULL|           NULL|
|    100005|    Brandon|    Myers|1988-07-01|2016-08-25| 4215| 75505.0|         NULL|           NULL|
|    100006|  Alexandra|  Hancock|1987-10-05|2025-01-20| 4865|147092.0|         NU

In [67]:
# ------------------------------------------------------------
# 18. Save DataFrame as Spark SQL Table
# ------------------------------------------------------------
emp_final.write.format("parquet").mode("overwrite").saveAsTable("emp_final")

In [68]:

# ------------------------------------------------------------
# 19. Read Data from Saved Table
# ------------------------------------------------------------
emp_new = spark.sql("SELECT * FROM emp_final")

# Reads saved Spark table.

In [69]:
# ------------------------------------------------------------
# 20. Show Table Data
# ------------------------------------------------------------
emp_new.show()

# Displays stored table records.


+----------+-----------+---------+----------+----------+-----+--------+-------------+---------------+
|first_name|  last_name|job_title|       dob|     email|phone|  salary|department_id|department_name|
+----------+-----------+---------+----------+----------+-----+--------+-------------+---------------+
|    100000|Christopher|Wilkerson|2004-05-01|2022-05-30| 3167| 62385.0|         NULL|           NULL|
|    100001|   Jennifer|    Moore|1992-11-24|2025-12-06| 9271| 87074.0|         NULL|           NULL|
|    100002|     Julian|    Jones|1992-09-23|2019-10-25| 9654|138568.0|         NULL|           NULL|
|    100003|     Thomas|  Roberts|1964-08-26|2018-04-28| 5948| 71807.0|         NULL|           NULL|
|    100004|     Robert|    Noble|1987-02-08|2011-05-02| 5094|108778.0|         NULL|           NULL|
|    100005|    Brandon|    Myers|1988-07-01|2016-08-25| 4215| 75505.0|         NULL|           NULL|
|    100006|  Alexandra|  Hancock|1987-10-05|2025-01-20| 4865|147092.0|         NU

In [70]:
# ------------------------------------------------------------
# 21. Describe Table Metadata
# ------------------------------------------------------------
spark.sql("DESCRIBE EXTENDED emp_final").show()

# Displays metadata such as:
# - Column names
# - Data types
# - Storage format
# - Table location
# - Provider
# - Statistics

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|          first_name|              string|   NULL|
|           last_name|              string|   NULL|
|           job_title|              string|   NULL|
|                 dob|              string|   NULL|
|               email|              string|   NULL|
|               phone|              string|   NULL|
|              salary|              double|   NULL|
|       department_id|                 int|   NULL|
|     department_name|              string|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|             default|       |
|               Table|           emp_final|       |
|               Owner|                root|       |
|        Created Time|Tue May 26 05:08:...|       |
|         La

# **Working with Dates**

In [71]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Create Spark Session
spark = SparkSession.builder.appName("DateFunctions").getOrCreate()

# Sample Data
data = [
    ("2025-07-20", "21/07/2025"),
    ("2025-08-15", "16/08/2025")
]

# Create DataFrame
df = spark.createDataFrame(data, ["date1", "date2"])

# Apply Date Functions
df2 = df.select(

    current_date().alias("Current_Date"),

    to_date(col("date1")).alias("Converted_Date1"),

    to_date(col("date2"), "dd/MM/yyyy").alias("Converted_Date2"),

    date_format(col("date1"), "dd-MM-yyyy").alias("Formatted_Date"),

    add_months(col("date1"), 2).alias("After_2_Months"),

    date_add(col("date1"), 10).alias("After_10_Days"),

    datediff(
        to_date(col("date2"), "dd/MM/yyyy"),
        to_date(col("date1"))
    ).alias("Date_Difference")
)

# Show Result
df2.show(truncate=False)





+------------+---------------+---------------+--------------+--------------+-------------+---------------+
|Current_Date|Converted_Date1|Converted_Date2|Formatted_Date|After_2_Months|After_10_Days|Date_Difference|
+------------+---------------+---------------+--------------+--------------+-------------+---------------+
|2026-05-26  |2025-07-20     |2025-07-21     |20-07-2025    |2025-09-20    |2025-07-30   |1              |
|2026-05-26  |2025-08-15     |2025-08-16     |15-08-2025    |2025-10-15    |2025-08-25   |1              |
+------------+---------------+---------------+--------------+--------------+-------------+---------------+



In [73]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Create Spark Session
spark = SparkSession.builder \
    .appName("Advanced_Date_Functions") \
    .getOrCreate()

# Sample Employee Data
data = [
    (1001, "Aniket", "2022-01-15"),
    (1002, "Naina", "2023-03-20"),
    (1003, "Pratap", "2021-11-10"),
    (1004, "Sagun", "2024-02-05")
]

# Create DataFrame
df = spark.createDataFrame(data, ["Emp_ID", "Emp_Name", "Joining_Date"])

# Convert Joining_Date to DateType
df = df.withColumn(
    "Joining_Date",
    to_date(col("Joining_Date"), "yyyy-MM-dd")
)

# Apply Advanced Date Functions
result_df = df.select(

    col("Emp_ID"),
    col("Emp_Name"),
    col("Joining_Date"),

    # Months worked till today
    round(
        months_between(current_date(), col("Joining_Date")),
        2
    ).alias("Months_Worked"),

    # Next Monday after joining
    next_day(col("Joining_Date"), "Mon").alias("Next_Monday"),

    # Truncate to Year
    trunc(col("Joining_Date"), "year").alias("Year_Start"),

    # Truncate to Month
    trunc(col("Joining_Date"), "month").alias("Month_Start"),

    # Extract Year
    year(col("Joining_Date")).alias("Joining_Year"),

    # Extract Quarter
    quarter(col("Joining_Date")).alias("Quarter"),

    # Extract Month
    month(col("Joining_Date")).alias("Month"),

    # Extract Day of Week
    dayofweek(col("Joining_Date")).alias("Day_Of_Week")
)

# Show Result
result_df.show(truncate=False)

+------+--------+------------+-------------+-----------+----------+-----------+------------+-------+-----+-----------+
|Emp_ID|Emp_Name|Joining_Date|Months_Worked|Next_Monday|Year_Start|Month_Start|Joining_Year|Quarter|Month|Day_Of_Week|
+------+--------+------------+-------------+-----------+----------+-----------+------------+-------+-----+-----------+
|1001  |Aniket  |2022-01-15  |52.35        |2022-01-17 |2022-01-01|2022-01-01 |2022        |1      |1    |7          |
|1002  |Naina   |2023-03-20  |38.19        |2023-03-27 |2023-01-01|2023-03-01 |2023        |1      |3    |2          |
|1003  |Pratap  |2021-11-10  |54.52        |2021-11-15 |2021-01-01|2021-11-01 |2021        |4      |11   |4          |
|1004  |Sagun   |2024-02-05  |27.68        |2024-02-12 |2024-01-01|2024-02-01 |2024        |1      |2    |2          |
+------+--------+------------+-------------+-----------+----------+-----------+------------+-------+-----+-----------+

